In [1]:
import cudaq
import math

# Rotation synthesis is randomized; seed it so that the counts are reproducible.
cudaq.set_random_seed(42)

/usr/local/cudaq/cudaq/__init__.py:282: UserWarning: cudaq-logical is in preview. Its APIs, behavior, and documentation may change substantially in upcoming versions.
  from . import logical  # noqa: F401
/tmp/ipykernel_3207/2008718347.py:1: FutureWarning: The CUDA-Q `sample` and `observe` algorithmic primitives will change in a future release. Existing code may require updates. See https://nvidia.github.io/cuda-quantum/latest/using/migration/upcoming_changes.html for details.
  import cudaq


# Defining targets and customizing compilation

## Compiling for built-in targets

Consider the circuit below. It is a simple 8-qubit circuit made of alternating layers of `rz` and entangling `cx` gates. Trotter steps are often used to construct circuits that simulate Hamiltonians, but there is no need to understand this for the purposes of this tutorial.

In [2]:
@cudaq.kernel
def trotter_steps(q: cudaq.qview):
    # Four Trotter steps of an 8-qubit Ising-like chain.
    for step in range(4):
        scale = step + 1
        for i in range(8):
            rz(math.pi / (8 * scale), q[i])
        for left in range(0, 7, 2):
            x.ctrl(q[left], q[left + 1])
            rz(math.pi / (16 * scale), q[left + 1])
            x.ctrl(q[left], q[left + 1])
        for left in range(1, 7, 2):
            x.ctrl(q[left], q[left + 1])
            rz(math.pi / (12 * scale), q[left + 1])
            x.ctrl(q[left], q[left + 1])


@cudaq.kernel
def trotter():
    q = cudaq.qvector(8)
    trotter_steps(q)
    mz(q)

At the most basic level, a compiler's job is to take this high-level description of a computation and produce a
circuit that a given target is able to execute.
Setting for instance the target to `iqm` will cause the compiler to produce a circuit that uses the basis gates
available on IQM's hardware:

In [3]:
cudaq.set_target("iqm", emulate=True)

results = cudaq.estimate(trotter)
print(results.resources)

[2026-11-05 19:19:14.466] [warning] [IQMServerHelper.cpp:573] Leaving %QPU_ARCH% unresolved: Unable to get quantum architecture for "default" from "http://localhost/": HTTP GET Error - status code 0: Failed to connect to localhost:80 after 0 ms: Could not connect to server: . Set IQM_QPU_QA or pass --mapping-file to supply it offline.


Path %QPU_ARCH% does not exist
Path %QPU_ARCH% does not exist


Total # of gates: 468, total # of qubits: 8, circuit depth: 103, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  cz :  56
  phased_rx :  404
}



As you can see from the resource estimates, the resulting circuit only contains `cz` and `phased_rx`
gates, alongside measurements.

> This below should be an 'info' block
Notice that when running the code above, CUDA-Q will warn you that it is unable to obtain a copy of the device architecture, which it would normally use for routing the circuit. You can fix the warning by either providing a valid IQM key or
passing a `mapping_file` directly as an option to `cudaq.set_target`. See [IQM Backend](../hardware/superconducting.html#iqm) for more information.

The compiler also supports synthesizing circuits to discrete gate sets, often required to encode
circuits using quantum error correcting codes.
We can for instance select CUDA-Q's built-in Clifford+T target to express the same kernel in terms of
Cliffords (C-NOTs, H, S, X, Z) and T gates.

In [4]:
CLIFFORD_T_TARGET = "compiler-bench-ftqc-clifford-t"
cudaq.set_target(CLIFFORD_T_TARGET)

results = cudaq.estimate(trotter)
print(results.resources)

Total # of gates: 3714, total # of qubits: 8, circuit depth: 754, multi-Q gate count: 14, multi-Q depth: 4
{ 
  mz :  8
  s :  674
  t :  1492
  h :  1507
  x :  8
  cx :  14
  z :  11
}



Note that as this gate set is discrete, the resulting circuit can only implement arbitrary rotation 
anlges up to a certain approximation error (by default 1e-10). The cudaq-logical package allows you to configure the precision of the approximation, see below.

## Controlling the optimization level of the compiler

The simplest configuration option that can be modified to adjust the behavior of the compiler is the optimization level. This is a number between 0 and 3 that can be passed to any of the CUDA-Q kernel launch functions and
that controls how aggressive the optimizations that the compiler applies should be. Higher levels enable better
optimizations, which can increase the compilation time but will result in fewer gates to be executed on the
quantum device.

> This should be an info box: advanced compiler optimizations (optimization_level > 1) are
currently only enabled by default for targets using the Clifford+T gate set. Support for
optimizations on noisy hardware and simulators is still experimental and can be enabled
using custom targets, see below.

Depending on the compile target, the compiler will prioritize different metrics and
may optimize for different cost functions.
Using the previous Clifford+T target, we can see a 75% reduction in T gate count as the
optimization level increases:

In [5]:
cudaq.set_target(CLIFFORD_T_TARGET)

for level in (0, 1, 2, 3):
    resources = cudaq.estimate(trotter, optimization_level=level).resources
    print(f'optimization_level = {level}:\n{resources}\n')

optimization_level = 0:
Total # of gates: 14926, total # of qubits: 8, circuit depth: 2990, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  x :  16
  cx :  56
  s :  2845
  t :  5978
  h :  6023
}


optimization_level = 1:
Total # of gates: 14172, total # of qubits: 8, circuit depth: 2990, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  x :  16
  cx :  56
  s :  2697
  t :  5676
  h :  5719
}


optimization_level = 2:
Total # of gates: 3714, total # of qubits: 8, circuit depth: 754, multi-Q gate count: 14, multi-Q depth: 4
{ 
  mz :  8
  s :  674
  t :  1492
  h :  1507
  x :  8
  cx :  14
  z :  11
}


optimization_level = 3:
Total # of gates: 3714, total # of qubits: 8, circuit depth: 754, multi-Q gate count: 14, multi-Q depth: 4
{ 
  mz :  8
  s :  674
  t :  1492
  h :  1507
  x :  8
  cx :  14
  z :  11
}




A Clifford+T target is also shipped with the cudaq-logical package, CUDA-Q's quantum error correction toolkit.  It comes with a simple
constructor to override the desired approximation precision: 

In [6]:
import cudaq.logical as cql

cudaq.set_target(cql.targets.clifford_t_target(precision=1e-10))
for level in range(3):
    estimate = cudaq.estimate(trotter, optimization_level=level)
    print(f"level {level}: {estimate.resources.count('t'):>5} T")

level 0:  5978 T
level 1:  5676 T
level 2:  1492 T


## Using custom compile targets for finer-grained compilation control

> This is an info box. From here on, we will rely on 'experimental' APIs to control the compilation.
These APIs are likely to change significantly in upcoming releases and behavior will change. Please do not hesitate to open
an issue on [our GitHub repository](https://github.com/NVIDIA/cuda-quantum/issues) if you find issues,
would like to share suggestions or make feature requests.

Using `optimization_level` to tweak the compiler is a pretty coarse knob. You can get into more detail
by tweaking the built-in targets to suit your needs, or even defining your own custom compile targets from
scratch.

To start with, here is how to retrieve the full configuration of the built-in Clifford+T compile target we have been using:

In [7]:
from cudaq._experimental import get_compile_target

cudaq.set_target(CLIFFORD_T_TARGET)
clifford_t_target = get_compile_target()
print(clifford_t_target)

CompileTarget(stages=[CompilationStage(ir_output_profile=QuakeUnmappedProfile(basis_gates=['h', 's', 't', 'x', 'y', 'z', 'swap', 'x(1)', 'y(1)', 'z(1)'], approximation_tolerance=1e-10, cost_model=CostModel.t_count, unroll_loops=LoopUnrolling.all, max_unroll_iterations=1024), lowering=inferred)], codegen=Codegen('', post_codegen_passes=''))


The Clifford+T compile target is defined by a single compilation stage `clifford_t_target.stages[0]`. The stage is defined using an `ir_output_profile`, which characterizes the kinds of quantum circuits that the target supports.
We can for instance replace the allowed basis gates to C-NOTs, Hadamards and rotations.
As a result, the compiler no longer synthesizes the rotations present in the `trotter` kernel:

In [8]:
new_basis_gates = ['x(1)', 'h', 'rz']

stage = clifford_t_target.stages[0]
profile = stage.ir_output_profile
profile.basis_gates = new_basis_gates
stage.ir_output_profile = profile
clifford_t_target.stages = [stage]
print(clifford_t_target)

# Pass a named target and a compile target: kernels run on the named target,
# but are compiled against the given compile target.
cudaq.set_target("qpp-cpu", compile_target=clifford_t_target)
for level in range(3):
    resources = cudaq.estimate(trotter, optimization_level=level).resources
    print(resources)

CompileTarget(stages=[CompilationStage(ir_output_profile=QuakeUnmappedProfile(basis_gates=['x(1)', 'h', 'rz'], approximation_tolerance=1e-10, cost_model=CostModel.t_count, unroll_loops=LoopUnrolling.all, max_unroll_iterations=1024), lowering=inferred)], codegen=Codegen('', post_codegen_passes=''))
Total # of gates: 124, total # of qubits: 8, circuit depth: 29, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  cx :  56
  rz :  60
}

Total # of gates: 121, total # of qubits: 8, circuit depth: 29, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  cx :  56
  rz :  57
}

Total # of gates: 37, total # of qubits: 8, circuit depth: 8, multi-Q gate count: 14, multi-Q depth: 4
{ 
  mz :  8
  cx :  14
  rz :  15
}



Note that a CUDA-Q target is made of a runtime component (_where_ should a kernel execute), and a compile component (_how_ a kernel should be compiled).
We only wish to override the compile part of the target.
This is why when using `cudaq.set_target`, we still specify a base built-in target (in our case, the QPP simulator): the runtime component remains unchanged
from the built-in, while the compile target gets overridden by our custom target.

However, as configured, our custom target is still configured to optimize T-count. This no longer makes sense now that we have
changed the gate set. We can instead define our own compile target, from scratch:

In [9]:
from cudaq._experimental import CompileTarget, QuakeUnmappedProfile, CostModel

nisq_gates = ["h", "rx", "rz", "z(1)"]
nisq_profile = QuakeUnmappedProfile(
    basis_gates=nisq_gates,
    cost_model=CostModel.two_qubit_gate_count)
nisq_target = CompileTarget(ir_output_profile=nisq_profile)
cudaq.set_target("qpp-cpu", compile_target=nisq_target)
for level in range(3):
    resources = cudaq.estimate(trotter, optimization_level=level).resources
    print(resources)

Total # of gates: 236, total # of qubits: 8, circuit depth: 54, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  cz :  56
  h :  112
  rz :  60
}

Total # of gates: 233, total # of qubits: 8, circuit depth: 54, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  cz :  56
  h :  112
  rz :  57
}

Total # of gates: 191, total # of qubits: 8, circuit depth: 45, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  rz :  15
  cz :  56
  h :  112
}



TODO: The results are actually worse than when optimizing for T-count!!

## Customizing all the way

Finally, to get maximal control over the compiler, users can choose to bypass the CUDA-Q pre-defined compilation options altogether 
and provide a custom MLIR compilation pipeline.

Other than for experimentation, we strongly advise users against this. The internal CUDA-Q compiler pipelines change over time. A pipeline
that works well today may no longer do so with the next release. What's even more likely (and worse in a way) is that your custom pipeline
continues to work but silently worsens over time. A fixed pipeline will quickly
become outdated and will lock you out of the regular performance improvements that ship with each CUDA-Q release.

In [10]:
from cudaq._experimental import CustomPipelineLowering

# just decompose the gates and do nothing else
custom_pipeline = "decomposition{basis=" + ",".join(nisq_gates) + "}"
custom_target = CompileTarget(custom_pipeline=CustomPipelineLowering(mid_level=custom_pipeline))
cudaq.set_target("qpp-cpu", compile_target=custom_target)

resources = cudaq.estimate(trotter).resources
print(resources)

Total # of gates: 236, total # of qubits: 8, circuit depth: 54, multi-Q gate count: 56, multi-Q depth: 16
{ 
  mz :  8
  cz :  56
  h :  112
  rz :  60
}

